## Part 4 - YouTube Content RAG Chatbot (Mini Project)

In [22]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings,ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


In [23]:
#Document Loader
yt_api = YouTubeTranscriptApi()
video_id = 'xAt1xcC6qfM'
transcripts = yt_api.fetch(video_id)

In [24]:
print(transcripts)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="what's the biggest misunderstanding", start=0.12, duration=3.799), FetchedTranscriptSnippet(text='about you whenever you hear about', start=2.2, duration=3.48), FetchedTranscriptSnippet(text="somebody who's got you know ridiculous", start=3.919, duration=4.121), FetchedTranscriptSnippet(text='amounts of money their values are', start=5.68, duration=4.879), FetchedTranscriptSnippet(text='different than your values you should be', start=8.04, duration=5.08), FetchedTranscriptSnippet(text="concerned about their agenda what's your", start=10.559, duration=5.48), FetchedTranscriptSnippet(text="biggest Fe I'll be sad as my brain gets", start=13.12, duration=5.56), FetchedTranscriptSnippet(text='less capable that disappoints me tell me', start=16.039, duration=4.48), FetchedTranscriptSnippet(text='one Behavior we all should adopt reading', start=18.68, duration=4.439), FetchedTranscriptSnippet(text="a lot being a student that's a big 

In [25]:
transcripts_text = " ".join(doc.text for doc in transcripts)

In [26]:
transcripts_text

"what's the biggest misunderstanding about you whenever you hear about somebody who's got you know ridiculous amounts of money their values are different than your values you should be concerned about their agenda what's your biggest Fe I'll be sad as my brain gets less capable that disappoints me tell me one Behavior we all should adopt reading a lot being a student that's a big part of My Success richest and most powerful men in the world Bill Gates has Unleashed a technological Revolution that has changed our lives Bill Gates of 25 and Bill Gates of 17 any change that you feel personally in my 20s being a maniac was the right thing my competitors would say oh no you work too hard and I'd say yes I do if you're in a race your 20s when you have no wife and no children that's the time to do it if you get an opportunity to invite three Indians for dinner who would that be there was a mathematician ramanujan I would have loved to have met him why do you think India is becoming a Global T

In [27]:
# Text Splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks = splitter.split_text(transcripts_text)

In [28]:
len(chunks)

75

In [29]:
#Embeddings
embedding = OllamaEmbeddings(model='nomic-embed-text')

# FAISS
vector_store = FAISS.from_texts(chunks,embedding)

In [30]:
vector_store.similarity_search('AI',k=3)

[Document(id='e284ef12-8209-4d48-b6ea-6c16279771dc', metadata={}, page_content="work yes AI is going to change your world and AI hopefully can help you uh do things faster and better so my prescription would be yes to use it for the areas that you're excited about any area that you're excited about learn from learn about that and try to make a bridge between Ai and that industry right I mean the internet has so much educational material on it so between all that material uh and then an AI that can help you know take long documents and you can have a dialogue with it uh you"),
 Document(id='4c1ba382-0716-4583-9181-626e1ef4c89a', metadata={}, page_content="are going to be available uh to everyone and so you ought to be a user of AI but the underlying stuff you know that's a pretty narrow set of people who will have an opportunity to uh you know push on a new Training Method um and even people who grew up with software some of them don't really get this because it's it's a bit more mathem

In [31]:
retriever = vector_store.as_retriever(search_kwargs={'k':3})

In [32]:
retriever.invoke('AI')

[Document(id='e284ef12-8209-4d48-b6ea-6c16279771dc', metadata={}, page_content="work yes AI is going to change your world and AI hopefully can help you uh do things faster and better so my prescription would be yes to use it for the areas that you're excited about any area that you're excited about learn from learn about that and try to make a bridge between Ai and that industry right I mean the internet has so much educational material on it so between all that material uh and then an AI that can help you know take long documents and you can have a dialogue with it uh you"),
 Document(id='4c1ba382-0716-4583-9181-626e1ef4c89a', metadata={}, page_content="are going to be available uh to everyone and so you ought to be a user of AI but the underlying stuff you know that's a pretty narrow set of people who will have an opportunity to uh you know push on a new Training Method um and even people who grew up with software some of them don't really get this because it's it's a bit more mathem

In [33]:
retriever.invoke('How to start a company?')

[Document(id='e13a932e-b1e3-4120-b5f9-2eaf50fae4e8', metadata={}, page_content="and what the first step you'll take well you'd mostly start a company because you think you enjoy working on something and you think you have a worldclass understanding of you know a unique contribution you know for me that was software for this day and age you know if you think you understand you know Fusion uh you know that's great if you're successful which uh that's risky um you know you could really improve the world you know AI is sort of the today's equivalent of what I did when I was"),
 Document(id='1e99f438-ea64-418f-ac55-808c8a372210', metadata={}, page_content="um you know we had a great 25th uh uh 25 year celebration and this great sitar artist you know came and uh that was a pretty special thing uh so it' be hard for me to pick there's there's uh so many incredible talents you can't pick three I I I think I defend uh some people by not including them uh fair fair okay you were talking about sm

In [34]:
prompt = PromptTemplate(template="""
You are a helpful assistant. Answer from the following context. If the context is insufficient then directly say 'I don't Know'
{context}
Question: {question}
""",
input_variables=['context','question']
)

In [ ]:
question = input('Enter The Question?') # How to become rich?
context = retriever.invoke(question)
context = " ".join(i.page_content for i in context)

In [43]:
context

"uh a great upbringing in education but you know less than 1% of the uh the total wealth um because I decided it wouldn't be a favor to them you know it's not a dynasty you know I'm not asking them to run Microsoft I want to give them a chance to have their own earnings and success you know be significant and not overshadowed by the incredible luck and good fortune I had and different families see that differently I think the people who've made uh fortunes from technology are less dynastic uh and you know it partly comes from having great universities uh you know not just the iits but starting with them um you know that's using the incredible Talent that's here so talking about Indian Talent you do a lot of philanthropy work here and you've met a lot of talented people here lot of rich people here as well do you and you believe that all the money that rich people have or like people who have made wealth for themselves they should use it for charity work they should pledge it in what's 

In [44]:
final_prompt = prompt.invoke({'question':question,'context':context})

In [45]:
LLM = ChatOllama(model='llama3.2')

In [46]:
LLM.invoke(final_prompt)

AIMessage(content='Becoming rich is not just about accumulating wealth, but also about using one\'s resources to make a positive impact on society. According to the context provided, the individual believes that people who have made their fortunes through technology, particularly those who have had access to great universities, are less likely to be seen as "dynastic" and are more likely to use their wealth for philanthropic purposes.\n\nAs for how to become rich, it seems that the individual emphasizes the importance of education and accessing great universities, which can provide access to incredible talent. However, it\'s also worth noting that their approach is more focused on giving people a chance to have their own earnings and success, rather than solely accumulating wealth for personal gain.\n\nSome key takeaways from this context might be:\n\n1. Education and access to great universities can provide a strong foundation for success.\n2. Philanthropy and using one\'s wealth for 

1. Ask at least 5 questions related to the video?

What is the main topic of the video?
What are the key concepts explained?
What are the advantages discussed in the video?
What examples are provided?
What is the main conclusion of the video?

2. Verify answers come from video content?

The chatbot retrieves relevant transcript chunks from the vector database and uses them as context to generate answers.

3. Handle unknown questions gracefully?
When asked an unrelated question, the chatbot responds: 'I don't Know'